# Lab - DT for classification of 3 classes

1. Use the **load_wine** data (remember to split your data into a train, validation, and test set). Find the optimal parameters (using the parameters discussed earlier). How high an accuracy can you achieve on the test data?

**See slides for more details!**

In [ ]:
import numpy as np
import pandas as pd

from sklearn import tree
from sklearn.datasets import load_wine
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

wine_data = load_wine(as_frame = True)

df = wine_data.frame.dropna()

y = df["target"]
X = df.drop(["target"], axis = 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size = 0.2, random_state = 42)

print(f"X_train {X_train.shape}, X_val {X_val.shape}, X_test {X_test.shape}, y_train {y_train.shape}, y_val {y_val.shape}, y_test {y_test.shape}")

X_train (113, 13), X_val (29, 13), X_test (36, 13), y_train (113,), y_val (29,), y_test (36,).


In [5]:
# Testing a default settings model over validation data.
dt_default = tree.DecisionTreeClassifier(random_state = 42).fit(X = X_train, y = y_train)

y_val_hat = dt_default.predict(X_val)

accuracy_default = accuracy_score(y_val, y_val_hat)
print(f"DT with default settings achieved {round(accuracy_default * 100, 1)}% validation accuracy.")

DT with default settings achieved 89.7% validation accuracy.


In [8]:
min_samples_split_list = [5, 6, 7, 8, 9, 10, 11, 12]
min_samples_leaf_list = [5, 6, 7, 8, 9, 10, 11, 12]
max_features_list = [5, 6, 7, 8, 9, 10, 11, 12]

results = []

for min_samples_split in min_samples_split_list:
    for min_samples_leaf in min_samples_leaf_list:
        for max_features in max_features_list:
            dt_current = tree.DecisionTreeClassifier(
                                min_samples_split = min_samples_split,
                                min_samples_leaf = min_samples_leaf,
                                max_features = max_features,
                                random_state = 42
                        ).fit(X = X_train, y = y_train)
            
            y_val_hat = dt_current.predict(X = X_test)

            accuracy = accuracy_score(y_true = y_test, y_pred = y_val_hat)

            results.append([accuracy, min_samples_split, min_samples_leaf, max_features])

results = pd.DataFrame(data = results, columns = ["Accuracy", "min_samples_split", "min_samples_leaf", "max_features"])
display(results)

,Accuracy,min_samples_split,min_samples_leaf,max_features
0,0.916667,5,5,5
1,0.861111,5,5,6
2,0.916667,5,5,7
3,0.944444,5,5,8
4,0.916667,5,5,9
...,...,...,...,...
507,0.916667,12,12,8
508,0.916667,12,12,9
509,0.916667,12,12,10
510,0.916667,12,12,11


In [10]:
results[results["Accuracy"] == results["Accuracy"].max()][:5]

,Accuracy,min_samples_split,min_samples_leaf,max_features
3,0.944444,5,5,8
6,0.944444,5,5,11
10,0.944444,5,6,7
12,0.944444,5,6,9
31,0.944444,5,8,12


In [19]:
dt_optimized = tree.DecisionTreeClassifier(
                min_samples_split = int(results.iloc[3]["min_samples_split"]),
                min_samples_leaf = int(results.iloc[3]["min_samples_leaf"]),
                max_features = int(results.iloc[3]["max_features"]),
                random_state = 42
            ).fit(X = pd.concat([X_train, X_val]), y = pd.concat([y_train, y_val]))

y_test_hat_optimized = dt_optimized.predict(X_test)

accuracy_optimized = accuracy_score(y_true = y_test, y_pred = y_test_hat_optimized)
print(f"Optimized DT achieved {round(accuracy_optimized * 100, 1)}% accuracy.")

Optimized DT achieved 94.4% accuracy.
